# Pipeline Nougat OCR para Google Colab (Resiliente)

Este cuaderno contiene un pipeline optimizado para procesar documentos PDF científicos y matemáticos complejos, convirtiéndolos a Markdown estructurado (`.mmd`), LaTeX compilable (`.tex`) y JSON para sistemas de recuperación RAG. Además, cuenta con recuperación automática de páginas vacías vía OCR Tesseract, fallback automático si falta Pandoc, y un visualizador interactivo premium de resultados.

In [ ]:
# @title 1. Instalación de Dependencias
!apt-get install -y pandoc tesseract-ocr tesseract-ocr-spa tesseract-ocr-eng
!pip install nougat-ocr pypdf torch tqdm transformers==4.38.2 albumentations==1.4.3 pypdfium2 fpdf2 pypandoc pypandoc-binary pytesseract
!python -m nltk.downloader words

import os
import json
import time
import shutil
import datetime
import hashlib
import re
import site
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB and not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Aviso: No se pudo montar Google Drive automáticamente: {e}")

In [ ]:
# @title 2. Verificación de GPU
import torch
import sys
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU Detectada: {device_name} (OK)")
    print("El sistema usará aceleración por hardware para el proceso de OCR.")
else:
    print("ERROR: No se detectó ninguna GPU en el entorno de ejecución.")
    print("Vaya a 'Entorno de ejecución' > 'Cambiar tipo de entorno de ejecución' y seleccione T4 GPU.")
    sys.exit("Por favor active una GPU para continuar con un rendimiento aceptable.")

In [ ]:
# @title 3. Aplicación de Parches de Compatibilidad
def apply_patches():
    sp_list = site.getsitepackages()
    if not sp_list: return
    sp = Path(sp_list[0])
    
    # Transformers
    for p in sp.rglob("transformers/configuration_utils.py"):
        content = p.read_text()
        if "PreTrainedConfig = PretrainedConfig" not in content:
            p.write_text(content + "\nPreTrainedConfig = PretrainedConfig\n")
            print("Patch Transformers: OK")

    # Nougat transforms
    for p in sp.rglob("nougat/transforms.py"):
        content = p.read_text()
        if 'from pydantic import BaseModel' in content and 'Field' not in content:
            content = content.replace('from pydantic import BaseModel', 'from pydantic import BaseModel, Field')
            content = content.replace('compression_type: str = "webp"', 'compression_type: str = Field("webp")')
            p.write_text(content)
            print("Patch Nougat Transforms: OK")

    # Rasterizer logic
    for p in sp.rglob("nougat/dataset/rasterize.py"):
        content = p.read_text()
        content = content.replace("pypdfium2.PdfDocument(pdf)", "pypdfium2.PdfDocument(str(pdf))")
        new_logic = """if hasattr(pdf, "render"):
            renderer = pdf.render(pypdfium2.PdfBitmap.to_pil, page_indices=pages, scale=dpi/72)
        elif hasattr(pdf, "render_topil"):
            renderer = pdf.render_topil(page_indices=pages, scale=dpi/72)
        else:
            renderer = [pdf[i].render(scale=dpi/72).to_pil() for i in pages]"""
        pattern = r"renderer\s*=\s*pdf\.render\(.*?\)"
        if "hasattr(pdf, \"render\")" not in content:
            import re
            content = re.sub(pattern, new_logic, content, flags=re.DOTALL)
            p.write_text(content)
            print("Patch Universal Rasterizer: OK")

apply_patches()

In [ ]:
# @title 4. Configuración del Pipeline
BASE_DIR = "/content/drive/MyDrive/NovaLibrary" # @param {type:"string"}
MODEL_SIZE = "0.1.0-small" # @param ["0.1.0-small", "0.1.0-base"]
FORCE_REPROCESS = False # @param {type:"boolean"}
LATEX_LANGUAGE = "spanish" # @param ["spanish", "english"]

# Verificación de Drive y fallback local (2.A)
actual_base_dir = Path(BASE_DIR)
if IN_COLAB:
    if not os.path.exists('/content/drive'):
        print("AVISO: Google Drive no está montado. Intentando montar...")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print(f"Error al montar Drive: {e}")
            
    if not os.path.exists('/content/drive'):
        actual_base_dir = Path("/content/NovaLibrary")
        print(f"⚠️ ADVERTENCIA: No se pudo acceder a Google Drive. Los archivos se guardarán LOCALMENTE en '{actual_base_dir}'.")
        print("⚠️ NOTA: Los archivos locales se perderán al desconectar la sesión de Colab. Descárguelos cuando termine.")
    else:
        print("✓ Google Drive montado correctamente.")

STRUCTURE = {
    "input": actual_base_dir / "input",
    "output": actual_base_dir / "output",
    "failed": actual_base_dir / "failed",
    "checkpoint": actual_base_dir / "checkpoint"
}

for p in STRUCTURE.values(): p.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = STRUCTURE["checkpoint"] / "registry.json"
LOG_PATH = STRUCTURE["checkpoint"] / "pipeline.log"

def log_message(msg):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a", encoding="utf-8") as f: f.write(f"[{ts}] {msg}\n")
    print(f"[{ts}] {msg}")

In [ ]:
class PipelineState:
    def __init__(self, path):
        self.path = path
        self.state = self._load()
    def _load(self):
        if self.path.exists():
            with open(self.path, "r", encoding="utf-8") as f: return json.load(f)
        return {"processed": {}, "failed": {}}
    def save(self):
        with open(self.path, "w", encoding="utf-8") as f: json.dump(self.state, f, indent=2, ensure_ascii=False)
    def is_processed(self, h): return h in self.state["processed"]
    def mark_success(self, h, name, out):
        self.state["processed"][h] = {"filename": name, "output": str(out), "ts": str(datetime.datetime.now())}
        self.save()
    def mark_failed(self, h, name, err):
        self.state["failed"][h] = {"filename": name, "error": str(err), "ts": str(datetime.datetime.now())}
        self.save()

def get_file_hash(path):
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192): sha.update(chunk)
    return sha.hexdigest()

In [ ]:
def extract_structured_data(mmd_path):
    print(f"Buscando estructuras en {mmd_path.name}...")
    with open(mmd_path, "r", encoding="utf-8") as f: content = f.read()
    p_inline = r"\\\(.*?\\\)"
    p_block = r"\\\[.*?\\\\]"
    equations = re.findall(f"{p_inline}|{p_block}", content, re.DOTALL)
    print(f"Ecuaciones detectadas: {len(equations)}")
    p_caption = r"\[caption\].*?\n"
    captions = re.findall(p_caption, content)
    
    sections = []
    current_hierarchy = ["Preliminares", "", "", ""]
    current_level = 1
    current_lines = []
    
    lines = content.replace("\r\n", "\n").split("\n")
    for line in lines:
        header_match = re.match(r'^(#{1,4})\s+(.*)$', line)
        if header_match:
            if current_lines:
                section_text = "\n".join(current_lines).strip()
                if section_text:
                    hierarchy_path = [h for h in current_hierarchy if h]
                    sections.append({
                        "title": current_hierarchy[current_level - 1],
                        "hierarchy": hierarchy_path,
                        "full_title": " > ".join(hierarchy_path),
                        "level": current_level,
                        "content": section_text,
                        "metrics": {
                            "characters": len(section_text),
                            "estimated_tokens": int(len(section_text.split()) * 1.3)
                        }
                    })
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_level = level
            current_hierarchy[level - 1] = title
            for i in range(level, 4):
                current_hierarchy[i] = ""
            current_lines = []
        else:
            current_lines.append(line)
            
    if current_lines:
        section_text = "\n".join(current_lines).strip()
        if section_text:
            hierarchy_path = [h for h in current_hierarchy if h]
            sections.append({
                "title": current_hierarchy[current_level - 1],
                "hierarchy": hierarchy_path,
                "full_title": " > ".join(hierarchy_path),
                "level": current_level,
                "content": section_text,
                "metrics": {
                    "characters": len(section_text),
                    "estimated_tokens": int(len(section_text.split()) * 1.3)
                }
            })
            
    print(f"Secciones identificadas: {len(sections)}")
    return {
        "metadata": {
            "source": mmd_path.name,
            "processed_at": str(datetime.datetime.now()),
            "equation_count": len(equations),
            "section_count": len(sections)
        },
        "equations": equations,
        "captions": captions,
        "sections": sections
    }

def save_structured_json(mmd_path):
    try:
        data = extract_structured_data(mmd_path)
        with open(mmd_path.with_suffix(".json"), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    except Exception as e: log_message(f"Error JSON: {e}")

def mmd_to_latex_fallback(mmd_content, title="Documento", language="spanish"):
    preamble = [
        "\\documentclass[11pt,a4paper]{article}",
        "\\usepackage[utf8]{inputenc}",
        "\\usepackage[T1]{fontenc}",
        "\\usepackage[" + language + "]{babel}",
        "\\usepackage{amsmath,amssymb,amsfonts}",
        "\\usepackage{graphicx}",
        "\\usepackage{geometry}",
        "\\geometry{margin=1in}",
        "\\title{ " + title + " }",
        "\\author{Pipeline Nougat OCR}",
        "\\date{\\today}",
        "\\begin{document}",
        "\\maketitle",
        "\\tableofcontents",
        "\\newpage"
    ]
    protected_math = []
    def save_math(m): protected_math.append(m.group(0)); return f"MATHPROTECT{len(protected_math)-1}Z"
    body = re.sub(r'\[MISSING_PAGE_EMPTY:\d+\]', '', mmd_content)
    body = re.sub(r'\[MISSING_PAGE_FAIL:\d+\]', r'\\begin{center}\\textbf{[ERROR: Página no procesada en el original]}\\end{center}', body)
    body = re.sub(r'\\\(.*?\\\)|\\\[.*?\\\\]|\\cite\{.*?\}|\\ref\{.*?\}|\\label\{.*?\}|\\begin\{.*?\}|\\end\{.*?\}', save_math, body, flags=re.DOTALL)
    raw_cmds_map = {
        r'\\section\*?\{([^{}]*)\}': r'LSECS\1LEND',
        r'\\subsection\*?\{([^{}]*)\}': r'LSUBS\1LEND',
        r'\\subsubsection\*?\{([^{}]*)\}': r'LSUBSUBS\1LEND',
        r'\\paragraph\*?\{([^{}]*)\}': r'LPARAGS\1LEND',
        r'\\subparagraph\*?\{([^{}]*)\}': r'LSTARTPAGS\1LEND',
        r'\\textbf\{([^{}]*)\}': r'LBOLDS\1LEND',
        r'\\textit\{([^{}]*)\}': r'LITALS\1LEND',
        r'\\underline\{([^{}]*)\}': r'LBOLDS\1LEND'
    }
    for _ in range(5):
        any_ch = False
        for rec, sub in raw_cmds_map.items():
            body, count = re.subn(rec, sub, body, flags=re.DOTALL)
            if count > 0: any_ch = True
        if not any_ch: break
    body = re.sub(r'^###### (.*)', r'LSTARTPAGS\1LEND', body, flags=re.MULTILINE)
    body = re.sub(r'^##### (.*)', r'LSTARTPAGS\1LEND', body, flags=re.MULTILINE)
    body = re.sub(r'^#### (.*)', r'LPARAGS\1LEND', body, flags=re.MULTILINE)
    body = re.sub(r'^### (.*)', r'LSUBSUBS\1LEND', body, flags=re.MULTILINE)
    body = re.sub(r'^## (.*)', r'LSUBS\1LEND', body, flags=re.MULTILINE)
    body = re.sub(r'^# (.*)', r'LSECS\1LEND', body, flags=re.MULTILINE)
    body = re.sub(r'\*\*(.*?)\*\*', r'LBOLDS\1LEND', body)
    body = re.sub(r'\*(.*?)\*', r'LITALS\1LEND', body)
    body = re.sub(r'_(.*?)_', r'LITALS\1LEND', body)
    special_chars = {'&': r'\\&', '%': r'\\%', '$': r'\\$', '_': r'\\_', '{': r'\\{', '}': r'\\}', '#': r'\\#', '~': r'\\textasciitilde{}', '^': r'\\textasciicircum{}'}
    for char, replacement in special_chars.items(): body = body.replace(char, replacement)
    body = body.replace("LSECS", r"\\section{").replace("LSUBSUBS", r"\\subsubsection{").replace("LSUBS", r"\\subsection{").replace("LPARAGS", r"\\paragraph{").replace("LSTARTPAGS", r"\\subparagraph{").replace("LBOLDS", r"\\textbf{").replace("LITALS", r"\\textit{").replace("LEND", "}")
    def restore_math(m): idx = int(m.group(1)); return protected_math[idx] if idx < len(protected_math) else m.group(0)
    body = re.sub(r'MATHPROTECT(\d+)Z', restore_math, body)
    pre = '\n'.join(preamble) + '\n'
    return pre + body + "\n\\end{document}"

def mmd_to_latex(mmd_content, title="Documento", language="spanish"):
    try:
        import pypandoc
        try:
            pypandoc.get_pandoc_path()
        except OSError:
            print("Pandoc no encontrado. Descargando versión interna...")
            pypandoc.download_pandoc()
        body = mmd_content
        body = re.sub(r'\[MISSING_PAGE_EMPTY:\d+\]', '', body)
        body = re.sub(r'\[MISSING_PAGE_FAIL:\d+\]', '\n\n**[ERROR: Página no procesada en el original]**\n\n', body)
        body = body.replace('\\(', '$').replace('\\)', '$')
        body = body.replace('\\[', '$$').replace('\\]', '$$')
        lang_map = {"spanish": "es", "english": "en"}
        lang_code = lang_map.get(language.lower(), language)
        latex_code = pypandoc.convert_text(
            body,
            to='latex',
            format='markdown',
            extra_args=[
                '--standalone',
                '-V', f'lang={lang_code}',
                '-V', f'title={title}',
                '-V', 'author=Pipeline Nougat OCR',
                '-V', 'geometry:margin=1in'
            ]
        )
        return latex_code
    except Exception as e:
        print(f"Fallo en conversión con Pandoc ({e}). Usando conversor de respaldo (Regex)...")
        return mmd_to_latex_fallback(mmd_content, title, language)

def recover_missing_pages(pdf_path, mmd_content, language="spanish"):
    missing_pages = re.findall(r'\[MISSING_PAGE_(EMPTY|FAIL):(\d+)\]', mmd_content)
    if not missing_pages: return mmd_content
    try:
        import pypdfium2 as pdfium
        import pytesseract
    except ImportError:
        print("Aviso: 'pytesseract' o 'pypdfium2' no disponibles. Saltando recuperación OCR.")
        return mmd_content
    tess_lang = "spa+eng" if language.lower() == "spanish" else "eng"
    modified_content = mmd_content
    try:
        src_pdf = pdfium.PdfDocument(str(pdf_path))
        for flag_type, pg_num_str in missing_pages:
            pg_idx = int(pg_num_str) - 1
            if pg_idx < 0 or pg_idx >= len(src_pdf): continue
            print(f"Recuperando página {pg_num_str} vía Tesseract OCR...")
            page = src_pdf[pg_idx]
            bitmap = page.render(scale=2)
            img = bitmap.to_pil()
            try:
                ocr_text = pytesseract.image_to_string(img, lang=tess_lang).strip()
            except Exception as ocr_err:
                print(f"No se pudo ejecutar Tesseract en la página {pg_num_str}: {ocr_err}")
                ocr_text = None
            if ocr_text:
                replacement = f"\n\n> [!NOTE]\n> **[PÁGINA {pg_num_str} RECUPERADA VÍA OCR TESSERACT]**\n>\n"
                indented_text = "\n".join([f"> {line}" for line in ocr_text.split("\n")])
                replacement += indented_text + "\n\n"
                target_tag = f"[MISSING_PAGE_{flag_type}:{pg_num_str}]"
                modified_content = modified_content.replace(target_tag, replacement)
                print(f"Página {pg_num_str} recuperada e inyectada con éxito.")
    except Exception as e: print(f"Error en recuperación de páginas: {e}")
    return modified_content

def generate_blank_page_report(pdf_path, mmd_content, out_p):
    try:
        import pypdfium2 as pdfium
        from fpdf import FPDF
        missing = re.findall(r'\[MISSING_PAGE_EMPTY:(\d+)\]', mmd_content)
        if not missing: return False
        temp_dir = Path("temp_audit_colab")
        temp_dir.mkdir(exist_ok=True)
        try:
            pdf = FPDF()
            src = pdfium.PdfDocument(str(pdf_path))
            for p in missing:
                pdf.add_page()
                pdf.set_font("Arial", size=12)
                pdf.cell(200, 10, txt=f"Pagina: {p}", ln=1)
                img = src[int(p)-1].render(scale=2).to_pil()
                img_p = temp_dir / f"page_{p}.png"
                img.save(img_p)
                pdf.image(str(img_p), x=10, y=30, w=190)
            pdf.output(str(out_p))
            return True
        finally:
            for f in temp_dir.glob("*.png"):
                try: os.remove(f)
                except: pass
            try: temp_dir.rmdir()
            except: pass
    except Exception as e: log_message(f"Error Auditor: {e}"); return False

In [ ]:
# @title 5. Ejecutar Pipeline
def main():
    state = PipelineState(REGISTRY_PATH)
    all_pdfs = list(STRUCTURE["input"].glob("*.pdf"))
    log_message(f"Encontrados {len(all_pdfs)} archivos.")
    
    for pdf_p in all_pdfs:
        h = get_file_hash(pdf_p)
        if state.is_processed(h) and not FORCE_REPROCESS:
            log_message(f"Saltando {pdf_p.name} (ya procesado).")
            continue
        
        log_message(f"--- Procesando: {pdf_p.name} ---")
        try:
            !nougat "{str(pdf_p)}" -o "{str(STRUCTURE['output'])}" --model {MODEL_SIZE} --no-skipping
            
            out_mmd = STRUCTURE["output"] / f"{pdf_p.stem}.mmd"
            if out_mmd.exists():
                with open(out_mmd, "r", encoding="utf-8") as f: mmd = f.read()
                
                # 1. Reporte de Auditoría de Vacíos (usar contenido crudo antes de recuperar)
                audit_p = STRUCTURE["output"] / f"{pdf_p.stem}_auditoria.pdf"
                generate_blank_page_report(pdf_p, mmd, audit_p)
                
                # 2. Recuperación Tesseract OCR
                log_message(f"Buscando páginas omitidas para recuperar en {pdf_p.name}...")
                recovered_mmd = recover_missing_pages(pdf_p, mmd, LATEX_LANGUAGE)
                if recovered_mmd != mmd:
                    with open(out_mmd, "w", encoding="utf-8") as f: f.write(recovered_mmd)
                    mmd = recovered_mmd
                
                # 3. Guardar JSON y LaTeX (con texto recuperado)
                save_structured_json(out_mmd)
                with open(out_mmd.with_suffix(".tex"), "w", encoding="utf-8") as f:
                    f.write(mmd_to_latex(mmd, pdf_p.stem, language=LATEX_LANGUAGE))
                
                state.mark_success(h, pdf_p.name, out_mmd)
                log_message("Éxito.")
            else:
                raise Exception("Error: El motor Nougat no generó el archivo de salida.")
        except Exception as e:
            log_message(f"Fallo: {e}")
            state.mark_failed(h, pdf_p.name, str(e))
            shutil.move(str(pdf_p), str(STRUCTURE["failed"] / pdf_p.name))

if __name__ == "__main__":
    main()